In [1]:
# pipeline_import.py
# ══════════════════════════════════════════════════════════════════════════════
# Script 1 : Pipeline import données macroéconomiques via Datastream (LSEG)
# Flux : metadata_excel/ ──► API Datastream ──► storage_excel/
# ══════════════════════════════════════════════════════════════════════════════

from pathlib import Path
from datetime import datetime
import pandas as pd
import lseg.data as ld


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

METADATA_DIR = Path(r"C:\Users\tdautheville\Downloads\METADATAPRIMAIRE")
STORAGE_DIR  = Path(r"\\tsclient\C\Users\tdautheville\OneDrive - Medef\Documents\Chambre\Portail de l'éco")

START_DATE = "2000-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

META_COLS_TO_KEEP = [
    "Name", "Symbol", "RIC", "Start Date", "End Date",
    "Category", "Market", "Source", "Unit", "Frequency",
    "Adjustment", "Dataset", "Full Name", "Activity"
]


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC 1 — LECTURE DES MÉTADONNÉES
# ══════════════════════════════════════════════════════════════════════════════
def load_metadata(filepath: Path) -> tuple[pd.DataFrame, str]:
    """
    Lit le fichier Excel métadonnée.
    Retourne le DataFrame nettoyé ET le nom de fichier de sortie
    construit depuis la colonne 'Full Name' (sans le pays devant la première virgule).
    """
    df = pd.read_excel(filepath)
    df.columns = [str(c).strip() for c in df.columns]

    if "RIC" not in df.columns:
        raise ValueError(f"Colonne 'RIC' absente dans {filepath.name}")

    cols = [c for c in META_COLS_TO_KEEP if c in df.columns]
    df   = df[cols].copy()
    df   = df.dropna(subset=["RIC"])
    df["RIC"] = df["RIC"].astype(str).str.strip()

    if "Activity" in df.columns:
        df = df[df["Activity"].str.strip().str.lower() == "active"]

    # ── Nommage automatique depuis Full Name ──────────────────────────────
    # Exemple : "Albania, Gross Domestic Product, Constant Prices, USD"
    # → on retire tout ce qui précède la première virgule + la virgule elle-même
    # → "Gross Domestic Product, Constant Prices, USD"
    # → on sanitize pour un nom de fichier Windows valide
    if "Full Name" in df.columns:
        raw_name = df["Full Name"].dropna().iloc[0]          # première valeur non-nulle
        # Coupe après la première virgule
        if "," in raw_name:
            raw_name = raw_name.split(",", maxsplit=1)[1].strip()
        # Retire les caractères interdits dans un nom de fichier Windows
        for char in r'\/:*?"<>|':
            raw_name = raw_name.replace(char, "")
        output_name = raw_name.strip()
    else:
        # Fallback : nom du fichier source
        output_name = filepath.stem

    return df.reset_index(drop=True), output_name


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC 2 — DÉTECTION DE LA FRÉQUENCE
# ══════════════════════════════════════════════════════════════════════════════
def detect_frequency(df_meta: pd.DataFrame, filepath: Path) -> str:
    """
    Détecte la fréquence depuis la colonne 'Frequency'.
    Libellés Datastream attendus : 'Monthly', 'Quarterly', 'Annual'.
    Retourne 'M', 'Q' ou 'A'.
    """
    if "Frequency" in df_meta.columns:
        freq_val = (
            df_meta["Frequency"]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .mode()[0]
        )
        if "monthly" in freq_val:
            return "M"
        if "quarterly" in freq_val:
            return "Q"
        if "annual" in freq_val:
            return "A"

    # Fallback nom de fichier
    name = filepath.stem.lower()
    if "monthly"   in name: return "M"
    if "quarterly" in name: return "Q"
    if "annual"    in name: return "A"

    print(f"  ⚠ Fréquence non détectée pour {filepath.name} — index brut conservé")
    return "UNKNOWN"


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC 3 — APPEL API DATASTREAM
# ══════════════════════════════════════════════════════════════════════════════
def fetch_data(df_meta: pd.DataFrame) -> pd.DataFrame:
    """
    Appelle ld.get_history() avec :
    - start = min(Start Date) du batch → remonte à la toute première donnée connue
    - end   = aujourd'hui
    La colonne 'Start Date' du fichier metadata Datastream contient la base date.
    """
    rics = df_meta["RIC"].tolist()

    # ── Start date dynamique ──────────────────────────────────────────────
    # On prend la plus ancienne Start Date du batch pour ne rien manquer
    if "Start Date" in df_meta.columns:
        start_date = (
            pd.to_datetime(df_meta["Start Date"], errors="coerce")
            .dropna()
            .min()
            .strftime("%Y-%m-%d")
        )
        print(f"   Start date (min batch) : {start_date}")
    else:
        # Fallback si colonne absente
        start_date = "1970-01-01"
        print(f"   ⚠ Colonne 'Start Date' absente — fallback {start_date}")

    try:
        df = ld.get_history(
            universe=rics,
            start=start_date,
            end=END_DATE,
        )
    except Exception as e:
        raise RuntimeError(f"Erreur API Datastream : {e}")

    if isinstance(df.columns, pd.MultiIndex):
        df = df["VALUE"]

    df.index = pd.to_datetime(df.index)
    df       = df.sort_index()
    df       = df.dropna(axis=1, how="all")

    if df.empty:
        raise ValueError("Aucune donnée retournée par l'API pour ces RICs.")

    return df


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC 4 — FORMATAGE DE L'INDEX TEMPOREL
# ══════════════════════════════════════════════════════════════════════════════
def format_index(df: pd.DataFrame, frequency: str) -> pd.DataFrame:
    """
    - M       → "Jan 2024", "Feb 2024"...
    - Q       → "Q1 2024", "Q2 2024"...
    - A       → "2024", "2023"...
    - UNKNOWN → index datetime brut
    """
    if frequency == "M":
        df.index = pd.to_datetime(df.index).strftime("%b %Y")
        df.index.name = "Month"

    elif frequency == "Q":
        df.index = df.index.to_period("Q").strftime("Q%q %Y")
        df.index.name = "Quarter"

    elif frequency == "A":
        df.index = df.index.to_period("A").strftime("%Y")
        df.index.name = "Year"

    else:
        df.index.name = "Date"

    return df


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC 5 — SAUVEGARDE EXCEL
# ══════════════════════════════════════════════════════════════════════════════

def save_to_storage(
    df_data:  pd.DataFrame,
    df_meta:  pd.DataFrame,
    filepath: Path,
    output_path: Path
) -> None:
    """
    Sauvegarde deux onglets dans un fichier Excel :
      - DATA     : séries temporelles (lignes = périodes, colonnes = RICs)
      - METADATA : métadonnées brutes du fichier source
    Le fichier de sortie porte le même nom que le fichier métadonnée source.
    """
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_data.to_excel(writer, sheet_name="DATA")
        df_meta.to_excel(writer, sheet_name="METADATA", index=False)

    print(f"  ✔ Sauvegardé : {output_path}")


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC 6 — ORCHESTRATION PAR FICHIER
# ══════════════════════════════════════════════════════════════════════════════
def process_one_file(filepath: Path) -> None:
    print(f"\n── Traitement : {filepath.name}")

    if not filepath.exists():
        raise FileNotFoundError(f"Fichier introuvable : {filepath}")

    # load_metadata retourne maintenant (df, output_name)
    df_meta, output_name = load_metadata(filepath)
    frequency = detect_frequency(df_meta, filepath)
    rics      = df_meta["RIC"].tolist()

    print(f"   Indicateur         : {output_name}")
    print(f"   Fréquence détectée : {frequency}")
    print(f"   RICs chargés       : {len(rics)}")
    print(f"   Aperçu RICs        : {rics[:5]}{'...' if len(rics) > 5 else ''}")

    # fetch_data reçoit maintenant df_meta pour extraire la start date
    df_data = fetch_data(df_meta)
    df_data = format_index(df_data, frequency)

    rics_ok = df_data.columns.tolist()
    rics_ko = [r for r in rics if r not in rics_ok]
    if rics_ko:
        print(f"  ⚠ RICs sans données : {rics_ko}")

    # Nom de fichier = indicateur sans pays, extension .xlsx
    output_path = STORAGE_DIR / f"{output_name}.xlsx"
    save_to_storage(df_data, df_meta, filepath, output_path)


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOC 7 — POINT D'ENTRÉE PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════
def main():
    STORAGE_DIR.mkdir(parents=True, exist_ok=True)

    # Récupère automatiquement tous les .xlsx du dossier dédié
    metadata_files = sorted(METADATA_DIR.glob("*.xlsx"))

    if not metadata_files:
        print("⚠ Aucun fichier .xlsx trouvé dans METADATAPRIMAIRE. Pipeline arrêté.")
        return

    print(f"{'═'*60}")
    print(f"  Pipeline import Datastream")
    print(f"  {datetime.today().strftime('%Y-%m-%d %H:%M')}")
    print(f"  Fichiers trouvés : {len(metadata_files)}")
    print(f"  Output           : {STORAGE_DIR}")
    print(f"{'═'*60}")

    for filepath in metadata_files:
        try:
            process_one_file(filepath)
        except Exception as e:
            print(f"  ✘ ERREUR sur {filepath.name} : {e}")

    print(f"\n{'═'*60}")
    print("  Import terminé.")
    print(f"{'═'*60}")


In [10]:
# BLOC 8 — Exécution (cellule finale notebook)
ld.open_session()
main()
ld.close_session()


════════════════════════════════════════════════════════════
  Pipeline import Datastream
  2026-06-11 12:47
  Fichiers trouvés : 44
  Output           : \\tsclient\C\Users\tdautheville\OneDrive - Medef\Documents\Chambre\Portail de l'éco
════════════════════════════════════════════════════════════

── Traitement : 639167692683670781.xlsx
   Indicateur         : Gross Domestic Product, Standardized, Constant Prices, SA, USD, 2010 Prices
   Fréquence détectée : Q
   RICs chargés       : 129
   Aperçu RICs        : ['aALCGDPD/CA', 'aAOCGDPD/CA', 'aARCGDPD/CA', 'aAUCGDPD/CA', 'aATCGDPD/CA']...
   Start date (min batch) : 1947-01-01


C:\Users\tdautheville\AppData\Roaming\Python\Python313\site-packages\lseg\data\_tools\_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  ✔ Sauvegardé : \\tsclient\C\Users\tdautheville\OneDrive - Medef\Documents\Chambre\Portail de l'éco\Gross Domestic Product, Standardized, Constant Prices, SA, USD, 2010 Prices.xlsx

── Traitement : 639167692958369802.xlsx
   Indicateur         : Consumer Spending, Standardized, Constant Prices, SA, USD, 2010 Prices
   Fréquence détectée : Q
   RICs chargés       : 101
   Aperçu RICs        : ['aARCPCED/CA', 'aAUCPCED/CA', 'aATCPCED/CA', 'aBHCPCED/CA', 'aBDCPCED/CA']...
   Start date (min batch) : 1947-01-01


C:\Users\tdautheville\AppData\Roaming\Python\Python313\site-packages\lseg\data\_tools\_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  ✔ Sauvegardé : \\tsclient\C\Users\tdautheville\OneDrive - Medef\Documents\Chambre\Portail de l'éco\Consumer Spending, Standardized, Constant Prices, SA, USD, 2010 Prices.xlsx

── Traitement : 639167693344602919.xlsx
   Indicateur         : Government Consumption, Standardized, Constant Prices, SA, USD, 2010 Prices
   Fréquence détectée : Q
   RICs chargés       : 101
   Aperçu RICs        : ['aARCGCED/CA', 'aAUCGCED/CA', 'aATCGCED/CA', 'aBHCGCED/CA', 'aBDCGCED/CA']...
   Start date (min batch) : 1949-01-01


...

  ✔ Sauvegardé : \\tsclient\C\Users\tdautheville\OneDrive - Medef\Documents\Chambre\Portail de l'éco\Trade Weighted Nominal Exchange Rate, % Month On Month, Standardized, Change Period Over Period.xlsx

── Traitement : 639167707052007808.xlsx
   Indicateur         : Consumer Price Index, Standardized, SA, Index, 2010 = 100
   Fréquence détectée : M
   RICs chargés       : 144
   Aperçu RICs        : ['aALCCPIE/CA', 'aDZCCPIE/CA', 'aAOCCPIE/CA', 'aAMCCPIE/CA', 'aAUCCPIE/CA']...
   Start date (min batch) : 1914-01-01


C:\Users\tdautheville\AppData\Roaming\Python\Python313\site-packages\lseg\data\_tools\_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  ⚠ RICs sans données : ['aBZCCPICUE']
  ✔ Sauvegardé : \\tsclient\C\Users\tdautheville\OneDrive - Medef\Documents\Chambre\Portail de l'éco\Consumer Price Index, Standardized, SA, Index, 2010 = 100.xlsx

── Traitement : 639167707351593104.xlsx
   Indicateur         : Consumer Price Index, Standardized, Index, 2010 = 100
   Fréquence détectée : M
   RICs chargés       : 144
   Aperçu RICs        : ['aALCCPIF/C', 'aDZCCPIF/C', 'aAOCCPIF/C', 'aAMCCPIF/C', 'aAUCCPIF/C']...
   Start date (min batch) : 1913-01-01


C:\Users\tdautheville\AppData\Roaming\Python\Python313\site-packages\lseg\data\_tools\_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  ⚠ RICs sans données : ['aBZCCPILJF']
  ✘ ERREUR sur 639167707351593104.xlsx : [WinError 1167] Le périphérique n’est pas connecté: "\\\\tsclient\\C\\Users\\tdautheville\\OneDrive - Medef\\Documents\\Chambre\\Portail de l'éco"

── Traitement : 639167707905250974.xlsx
  ✘ ERREUR sur 639167707905250974.xlsx : [WinError 1167] Le périphérique n’est pas connecté: "\\\\tsclient\\C\\Users\\tdautheville\\OneDrive - Medef\\Documents\\Chambre\\Portail de l'éco"

── Traitement : 639167708492086335.xlsx
  ✘ ERREUR sur 639167708492086335.xlsx : [WinError 1167] Le périphérique n’est pas connecté: "\\\\tsclient\\C\\Users\\tdautheville\\OneDrive - Medef\\Documents\\Chambre\\Portail de l'éco"

── Traitement : 639167709104984637.xlsx
  ✘ ERREUR sur 639167709104984637.xlsx : [WinError 995] L’opération d’entrée/sortie a été abandonnée en raison de l’arrêt d’un thread ou à la demande d’une application: "\\\\tsclient\\C\\Users\\tdautheville\\OneDrive - Medef\\Documents\\Chambre\\Portail de l'éco"

── Traite